# A Spam Classifier
The work never ends! We must practice however, we are going to make a spam classifier here, 
hopefully I can create this in less time than the last project with the time we have.

This time though, our teacher has given us instructions. They go as follows:

```python
# 1. Download examples of spam and ham from Apache SpamAssassin’s public datasets.

# 2. Unzip the datasets and familiarize yourself with the data format.

# 3. Split the datasets into a training set and a test set.

# 4. Write a data preparation pipeline to convert each email into a feature vector. Your preparation 
#   pipeline should transform an email into a (sparse) vector that indicates the presence or absence 
#   of each possible word. For example, if all emails only ever contain four words, “Hello,” “how,” 
#   “are,” “you,” then the email “Hello you Hello Hello you” would be converted into a vector [1, 0, 
#   0, 1] (meaning [“Hello” is present, “how” is absent, “are” is absent, “you” is present]), or [3, 
#   0, 0, 2] if you prefer to count the number of occurrences of each word.

# 5. You may want to add hyperparameters to your preparation pipeline to control whether or not to 
#   strip off email headers, convert each email to lowercase, remove punctuation, replace all URLs 
#   with “URL,” replace all numbers with “NUMBER,” or even perform stemming (i.e., trim off word 
#   endings; there are Python libraries available to do this).

# 6. Finally, try out several classifiers and see if you can build a great spam classifier, with 
#   both high recall and high precision
```

## Setup

In [1]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

extract_files = False

In [2]:
# get ham_2
import tarfile

with tarfile.open('20030228_easy_ham_2.tar.bz2', 'r:bz2') as f:
    f.list(verbose=True)
    if extract_files: f.extractall() # catch

FileNotFoundError: [Errno 2] No such file or directory: '20030228_easy_ham_2.tar.bz2'

In [ ]:
# get spam_2

import tarfile

with tarfile.open('20050311_spam_2.tar.bz2', 'r:bz2') as f:
    f.list(verbose=True)
    if extract_files: f.extractall()

### File Layout
These emails follow a similar structure. Below is a sample truncated email:
```python
'''
From exmh-workers-admin@redhat.com  Wed Aug 21 16:46:03 2002
Return-Path: <exmh-workers-admin@spamassassin.taint.org>
Delivered-To: yyyy@localhost.netnoteinc.com
Received: from localhost (localhost [127.0.0.1])
	by phobos.labs.netnoteinc.com (Postfix) with ESMTP id 7B74843C32
	for <jm@localhost>; Wed, 21 Aug 2002 11:46:03 -0400 (EDT)
{...truncated, more headers}
\n
Date: Wed, 21 Aug 2002 10:47:32 -0500

--==_Exmh_-2080822444P
Content-Type: text/plain; charset=us-ascii

> From:  Chris Garrigues <cwg-exmh@DeepEddy.Com>
> Date:  Wed, 21 Aug 2002 10:40:39 -0500
>
> > From:  Chris Garrigues <cwg-exmh@DeepEddy.Com>
{...truncated, more body}
_______________________________________________
Exmh-workers mailing list
Exmh-workers@redhat.com
https://listman.redhat.com/mailman/listinfo/exmh-workers
'''
```
Most emails follow this pattern of format... as such, we can derive the following structure:
1. **From:** `From <email> <date> <time> <year>`
	* note that there are scenarios where this From field is missing
2. **Headers:** `<header name>: data`
	* note that if a header is multi-line, it will continue with a newline and a tab
3. **Newline:** there is always a single newline empty line seperating the Headers from the Body
4. **Body:** any text


### Line Extraction

In [ ]:
# how to convert into a dataframe, lets look through a specific file to see how it reads.
with open('./easy_ham_2/00003.19be8acd739ad589cd00d8425bac7115', 'r') as f:
    for line in f:
        print(line, end='') # no /n needed since present

In [ ]:
# lets create a function that will split up an email into from, headers, body
def split_email(email):
    frm, headers, body = '', '', ''

    # verify first line from or header
    first_line: str = email.readline()

    # we can tell something is a header with the presence of ':' within the first word.
    if ':' not in first_line.split()[0]:
        frm = first_line.strip() + '\n'
    else:
        headers = first_line.strip() + '\n'

    # extract headers
    for header in email:
        # the body section begins when there is a single \n line.
        if header == '\n':
            break
        else:
            headers += header

    # extract body
    for line in email:
        body += line

    return (frm, headers, body)

In [ ]:
# test function 
test_frm = ()
with open('./easy_ham_2/00003.19be8acd739ad589cd00d8425bac7115', 'r') as f:
    test_frm = split_email(f)

print('---------------------FROM---------------------', test_frm[0], end='', sep='\n')
print('---------------------HEADERS---------------------', test_frm[1], end='', sep='\n')
print('---------------------BODY---------------------', test_frm[2], end='', sep='\n')

In [ ]:
# test on file without from field
test_no_frm = ()
with open('./easy_ham_2/00001.1a31cc283af0060967a233d26548a6ce', 'r') as f:
    test_no_frm = split_email(f)

print('---------------------FROM---------------------', test_no_frm[0], end='', sep='\n')
print('---------------------HEADERS---------------------', test_no_frm[1], end='', sep='\n')
print('---------------------BODY---------------------', test_no_frm[2], end='', sep='\n')

### Inner Line Extraction
How much further can we split up these lines? Lets start with the `from` line. It is apparent that 
all `from` lines are formatted as so:
```python
'''
From <email>  <weekday> <month> <day> <time> <year>
'''
```
- Note that there is a double space between the email and the rest of the information.
- Also note that, within emails without the `from` line, there is date information included as a
header at the bottom, we will discuss this soon.

In [ ]:
# from line extraction function
def extract_frm(frm: str):
    data = {
        'email': '',
        'weekday': '',
        'month': '',
        'day': '',
        'time': '',
        'year': ''
    }

    if frm:
        # [from, email, space, weekday, month, day, time, year\n]
        values = frm.split(' ')
        data['email'] = values[1]
        data['weekday'] = values[3]
        data['month'] = values[4]
        data['day'] = values[5]
        data['time'] = values[6]
        data['year'] = values[7][:-1] #note there is a newline present, we must remove.

    return data

extract_frm(test_frm[0])

Now onto headers. Most headers follow a standard format. With:
```python
'''
# regular header
<header name>: <header body>

# multi-line header
<header name>: <header body>
\t  <header body>
'''
```
Some headers like `Recieved` are repeated multiple times.

In [ ]:
def extract_headers(headers: str):
    '''
    Will create a dictionary containing each header and one or more bodies for each header. Each
    header title will contain a list of bodies, in order of appearance within an email (topmost
    first).
    '''

    result = {}
    # a new line will be included at the end of the headers.
    headers_by_line = headers.split('\n')[0:-1]

    header_index = 0
    curr_body = ''
    curr_header = ''
    for line in headers_by_line:
        
        # if \t present, it is a multiline 
        if ('\t' in line) or ('    ' in line):
            curr_body += line
        else:
            
            # save previous
            if result.get(curr_header):
                result[curr_header].append([curr_body, header_index])
            else:
                result[curr_header] = [ curr_body , header_index ]
            header_index += 1 

            # reassign
            l = line.split(':', 1)
            curr_header = l[0]
            curr_body = l[1][1:] # there will always be a single empty space between the seperator (:)

    # final save
    if result.get(curr_header):
        result[curr_header].append(curr_body)
    else:
        result[curr_header] = [ curr_body, header_index ]

    result.pop('') # cleanup, normally index 0 as well
    return result

extract_headers(test_frm[1])

There is much to be done within each header line, we can clean the result in many ways, things we
can replace are tabs (`\t`) or 4spaces (`    `). We should also replace email usernames with 
something like USERNAME, and URLS with URL. As well, it would be more beneficial to store each word
as a list instead of a complete string.

In [ ]:
def clean_line(line: str):
    '''
    Will perform the following operations on a given line:
    1. Remove tabs or 4spaces.
    2. Replace email usernames with USERNAME, preserve domain.
    3. Replace urls with URL.
    4. Return line as string of seperated values.
    '''
    # we are aiming for an O(n) time, therefore this will be fairly optimized.
    length = len(line)
    if length == 0: return 

    # remove tabs or spaces from line
    if length >= 4 and line [:4] == '\t\t':
        line = line[4:] # remove \t\t
    elif length >= 2 and line[:2] == '\t':
        line = line[2:] # remove \t
    elif length >= 2 and line[:4] == '    ':
        line = line[4:] # remove 4space
    
    line = line + ' ' # space for cleanup, will always append the last word

    words = []
    curr = ''
    # go character by character to figure out these things.
    # what do we look for?:
    #   1. keep track of first alphanumeric value.
    #   2. if we encounter a @, we know that the current string we are building since the first
    #      alphanumeric value should be replaced with USERNAME
    #   3. We can keep track of URLS with verifying if the address begins with http, and ending if
    #      a character is neither alphanumeric nor a forward slash.
    #   4. If we find a space, that means we should move on to the next word.
    #   5. Remove chevrons, commas, semicolons (<, >, ',', ;)
    http = ['h', 't', 't', 'p']
    web = 0 # keep track for matching
    for c in line:

        # web portion
        if web == 3:
            pass 
        elif c == http[web]:
            web += 1
        else:
            web = 0

        if c in ['<', '>', ',', ';']:
            continue
        elif c == '@':
            curr = 'USERNAME@' # replace username
        elif c == ' ':
            if web == 3: # full website
                words.append('URL')
            else:
                words.append(curr)
            curr = ''
            web = 0
        else:
            curr += c

    return words

        
clean_line('exmh-workers-admin@spamassassin.taint.org <mailto:exmh-workers-request@spamassassin.taint.org?subject=help> <https://listman.spamassassin.taint.org/mailman/listinfo/exmh-workers>, <mailto:exmh-workers-request@redhat.com?subject=unsubscribe>')


In [ ]:
def optimized_extract_headers(headers: str):
    '''
    Will create a dictionary containing each header and one or more bodies for each header. Each
    header title will contain a list of bodies, in order of appearance within an email (topmost
    first).
    '''

    result = {}
    # a new line will be included at the end of the headers.
    headers_by_line = headers.split('\n')[0:-1]

    header_index = 0
    curr_body = []
    curr_header = ''
    for line in headers_by_line:
        
        # if \t present, it is a multiline 
        if ('\t' in line) or ('    ' in line):
            curr_body += clean_line(line)
        else:
            
            # save previous
            if result.get(curr_header):
                result[curr_header].append([curr_body, header_index])
            else:
                result[curr_header] = [ curr_body , header_index ]
            header_index += 1 

            # reassign
            l = line.split(':', 1)
            curr_header = l[0]
            curr_body = clean_line(l[1][1:]) # there will always be a single empty space between the seperator (:)

    # final save
    if result.get(curr_header):
        result[curr_header].append(curr_body)
    else:
        result[curr_header] = [ curr_body, header_index ]

    result.pop('') # cleanup, normally index 0 as well
    return result

optimized_extract_headers(test_frm[1])